# 01 · Scraping

Descarga desde iNaturalist a `Insectos/<especie>/` y registra la procedencia en
`metadata.csv`. Incremental: deduplica por `photo_id`, así que reejecutar solo
baja lo que falta.

Solo licencias abiertas. Quedan fuera las fotos sin licencia declarada y
`cc-by-nc-nd`, cuyo ND prohíbe obras derivadas — recortar y aumentar lo es.

In [1]:
import hashlib
import json
import re
import time
from collections import Counter

import requests

from corpus import IMAGENES, LISTA_ESPECIES, METADATA, escribirCsv, leerCsv

API = "https://api.inaturalist.org/v1"
LICENCIAS = "cc0,cc-by,cc-by-nc,cc-by-sa,cc-by-nc-sa"
PAUSA = 0.25

POR_ESPECIE = 200
SOLO_ESPECIES = None          # {"eurybia_dardus"} para rellenar unas pocas
INCLUIR_NO_CONFIRMADAS = False

COLUMNAS = ["especie", "taxon_id", "observation_id", "photo_id", "quality_grade",
            "licencia", "atribucion", "url", "archivo", "sha256"]

## Cliente

In [2]:
def pedir(sesion, ruta, params):
    respuesta = sesion.get(f"{API}/{ruta}", params=params, timeout=30)
    respuesta.raise_for_status()
    return respuesta.json()


def nombreCorto(entrada):
    limpio = re.sub(r"\(.*?\)|,.*$", "", entrada)
    return " ".join(limpio.split()[:2])


def carpetaDe(nombre):
    return nombre.lower().replace(" ", "_")


def buscarTaxon(sesion, nombre):
    resultados = pedir(sesion, "taxa", {"q": nombre, "rank": "species",
                                        "per_page": 1, "is_active": "true"})["results"]
    return resultados[0] if resultados else None


def disponibilidad(sesion, taxonId):
    """Techo real de una especie, para saber si subir el tope sirve de algo."""
    return pedir(sesion, "observations", {
        "taxon_id": taxonId, "quality_grade": "research", "photos": "true",
        "photo_license": LICENCIAS, "per_page": 1,
    })["total_results"]


def descargar(sesion, url, destino):
    respuesta = sesion.get(url, timeout=45)
    respuesta.raise_for_status()
    destino.write_bytes(respuesta.content)
    return hashlib.sha256(respuesta.content).hexdigest()

## Paginación

`incluirNoConfirmadas` quita el filtro `quality_grade`: es la única fuente que
hace crecer a las especies escasas, que ya agotaron lo research-grade, a cambio
de identificaciones sin confirmar. Cada foto sale con su grado para poder
separarlas después.

In [3]:
def fotosDe(sesion, taxonId, maximo, incluirNoConfirmadas=False):
    vistas, pagina = set(), 1
    while len(vistas) < maximo:
        filtros = {"taxon_id": taxonId, "photos": "true", "photo_license": LICENCIAS,
                   "order_by": "id", "order": "asc", "per_page": 200, "page": pagina}
        if not incluirNoConfirmadas:
            filtros["quality_grade"] = "research"

        observaciones = pedir(sesion, "observations", filtros)["results"]
        if not observaciones:
            return
        for observacion in observaciones:
            for foto in observacion["photos"]:
                if foto["id"] in vistas or not foto.get("license_code"):
                    continue
                vistas.add(foto["id"])
                yield observacion["id"], observacion.get("quality_grade", ""), foto
                if len(vistas) >= maximo:
                    return
        pagina += 1

## Descarga

Acumula sobre `metadata.csv` y lo reescribe entero al final

In [5]:
especies = json.loads(LISTA_ESPECIES.read_text())["species"]
especies = [e for e in especies if "sp." not in e["name"]]
if SOLO_ESPECIES:
    especies = [e for e in especies if carpetaDe(nombreCorto(e["name"])) in SOLO_ESPECIES]

sesion = requests.Session()
sesion.headers["User-Agent"] = "butterfly-dataset/1.0"

filas = leerCsv(METADATA)
conocidas = {fila["photo_id"] for fila in filas}
print(f"{len(filas)} fotos ya registradas · {len(especies)} especies a recorrer")

for numero, entrada in enumerate(especies, start=1):
    nombre = nombreCorto(entrada["name"])
    taxon = buscarTaxon(sesion, nombre)
    time.sleep(PAUSA)
    if taxon is None:
        print(f"[{numero}/{len(especies)}] {nombre}: sin taxón")
        continue

    carpeta = IMAGENES / carpetaDe(nombre)
    carpeta.mkdir(parents=True, exist_ok=True)
    nuevas = 0
    for observacionId, grado, foto in fotosDe(sesion, taxon["id"], POR_ESPECIE,
                                              INCLUIR_NO_CONFIRMADAS):
        if str(foto["id"]) in conocidas:
            continue
        url = foto["url"].replace("square", "large")
        archivo = carpeta / f"{carpetaDe(nombre)}_{foto['id']}.jpg"
        filas.append({
            "especie": carpetaDe(nombre), "taxon_id": taxon["id"],
            "observation_id": observacionId, "photo_id": foto["id"],
            "quality_grade": grado, "licencia": foto["license_code"],
            "atribucion": foto.get("attribution", ""), "url": url,
            "archivo": archivo.name, "sha256": descargar(sesion, url, archivo),
        })
        nuevas += 1
        time.sleep(PAUSA)

    escribirCsv(METADATA, filas, COLUMNAS)   # por especie: interrumpir no deja imágenes sin procedencia
    print(f"[{numero}/{len(especies)}] {nombre}: +{nuevas}")

print(f"\n{len(filas)} fotos en metadata.csv")

14685 fotos ya registradas · 86 especies a recorrer


KeyboardInterrupt: 

## Inventario

In [ ]:
import pandas as pd

meta = pd.DataFrame(filas).drop_duplicates("photo_id")
porEspecie = meta.groupby("especie").agg(fotos=("photo_id", "size"),
                                         observaciones=("observation_id", "nunique"))
print(f"{len(porEspecie)} especies · {porEspecie.fotos.sum():,} fotos")
print(f"por especie: min {porEspecie.fotos.min()}, mediana "
      f"{porEspecie.fotos.median():.0f}, max {porEspecie.fotos.max()}")
porEspecie.sort_values("fotos").head(10)

## Cuánto más hay disponible

Antes de subir el tope conviene saber el techo: una especie que ya agotó
iNaturalist no crece por más que se scrapee.

In [ ]:
techos = {}
for especie, fila in porEspecie.iterrows():
    taxon = meta[meta.especie == especie].taxon_id.iloc[0]
    techos[especie] = disponibilidad(sesion, int(taxon))
    time.sleep(PAUSA)

porEspecie["obs_disponibles"] = pd.Series(techos)
porEspecie["fotos_estimadas"] = (porEspecie.obs_disponibles
                                 * porEspecie.fotos / porEspecie.observaciones).round()
for tope in (200, 350, 500):
    n = (porEspecie.fotos_estimadas >= tope).sum()
    print(f"  alcanzarían {tope} fotos: {n}/{len(porEspecie)} especies")
porEspecie.nsmallest(10, "fotos_estimadas")